# Understanding Qwen3-VL from Scratch

**A Karpathy-style deep dive into Vision-Language Models**

In this notebook, we'll build intuition for how Qwen3-VL works by examining every component from first principles. We'll trace tensors through the entire forward pass, annotating shapes and understanding *why* each operation exists.

---

## The Big Picture

Qwen3-VL is a **Vision-Language Model (VLM)** that can understand both images and text. The key insight is simple:

> **Images are just sequences of tokens, like text.**

We chop images into patches, project them into the same embedding space as text tokens, and feed everything into a transformer. That's it. The magic is in the details.

## Part 1: Setup and Imports

Let's start by importing what we need and setting up our environment.

In [ ]:
import sys
sys.path.insert(0, './tiny-qwen')

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 2: The Image → Patches Pipeline

This is where the magic begins. We need to convert a 2D image into a 1D sequence that a transformer can process.

### Step 1: Understanding Patch Extraction

An image of size `(H, W, 3)` gets divided into non-overlapping patches of size `16×16`. Each patch becomes a "token".

```
Image (672, 448, 3)  →  Patches (42×28 = 1176 patches)  →  Tokens (294 after 2×2 merge)
```

Why 294? Because we do a **spatial merge** of 2×2 patches → 1 token. This reduces sequence length by 4×.

In [ ]:
# ============================================================================
# PATCH EXTRACTION - The first step in converting images to tokens
# ============================================================================

# Key constants that define how we chop up images
SPATIAL_PATCH_SIZE = 16      # Each patch is 16×16 pixels
SPATIAL_MERGE_SIZE = 2       # We merge 2×2 patches into 1 token (reduces seq len by 4×)
TEMPORAL_PATCH_SIZE = 2      # For video: 2 frames per temporal patch (images duplicate)

def visualize_patch_extraction(image_path):
    """
    Visualize how an image gets divided into patches.
    This is EXACTLY what happens in the model's preprocessing.
    """
    # Load and get original dimensions
    img = Image.open(image_path)
    orig_w, orig_h = img.size
    
    # Step 1: Resize to be divisible by (patch_size × merge_size) = 32
    # This is crucial - we need clean divisions for patch extraction
    factor = SPATIAL_PATCH_SIZE * SPATIAL_MERGE_SIZE  # 16 × 2 = 32
    new_h = round(orig_h / factor) * factor
    new_w = round(orig_w / factor) * factor
    
    print(f"Original size: ({orig_h}, {orig_w})")
    print(f"Resized to: ({new_h}, {new_w}) - divisible by {factor}")
    
    # Step 2: Calculate grid dimensions
    grid_h = new_h // SPATIAL_PATCH_SIZE  # How many 16×16 patches vertically
    grid_w = new_w // SPATIAL_PATCH_SIZE  # How many 16×16 patches horizontally
    
    print(f"\nPatch grid: ({grid_h}, {grid_w}) = {grid_h * grid_w} patches")
    
    # Step 3: After spatial merge, we have fewer tokens
    merged_h = grid_h // SPATIAL_MERGE_SIZE
    merged_w = grid_w // SPATIAL_MERGE_SIZE
    final_tokens = merged_h * merged_w
    
    print(f"After 2×2 merge: ({merged_h}, {merged_w}) = {final_tokens} tokens")
    print(f"\n→ This image will add {final_tokens} tokens to the sequence!")
    
    # Visualize
    img_resized = img.resize((new_w, new_h))
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Original with patch grid overlay
    axes[0].imshow(img_resized)
    axes[0].set_title(f'16×16 Patches: {grid_h}×{grid_w} = {grid_h*grid_w} patches')
    for i in range(0, new_h + 1, 16):
        axes[0].axhline(y=i, color='white', linewidth=0.5, alpha=0.5)
    for j in range(0, new_w + 1, 16):
        axes[0].axvline(x=j, color='white', linewidth=0.5, alpha=0.5)
    
    # Merged grid
    axes[1].imshow(img_resized)
    axes[1].set_title(f'After 2×2 Merge: {merged_h}×{merged_w} = {final_tokens} tokens')
    for i in range(0, new_h + 1, 32):
        axes[1].axhline(y=i, color='yellow', linewidth=2)
    for j in range(0, new_w + 1, 32):
        axes[1].axvline(x=j, color='yellow', linewidth=2)
    
    plt.tight_layout()
    plt.show()
    
    return grid_h, grid_w, final_tokens

# Uncomment to test with an image:
# grid_h, grid_w, n_tokens = visualize_patch_extraction('path/to/your/image.jpg')

## Part 3: The Patch Embedding Layer (Conv3d)

Here's a beautiful trick: we use a **3D convolution** to convert patches into embeddings in one operation.

Why 3D? Because we handle video! The dimensions are `(temporal, height, width)`.

For images, we duplicate the frame to create a "fake" video of 2 identical frames.

In [ ]:
# ============================================================================
# PATCH EMBEDDING - Converting raw pixels to embeddings
# ============================================================================

class PatchEmbed(nn.Module):
    """
    The first layer of the vision encoder.
    
    Input:  (N, 3 × 2 × 16 × 16) = (N, 1536) flattened patches
    Output: (N, n_embed) embeddings
    
    Where:
    - N = total number of patches across all images
    - 3 = RGB channels
    - 2 = temporal patch size (frames)
    - 16 = spatial patch size
    """
    def __init__(self, n_embed=1280, patch_size=16, temporal_patch_size=2, in_channels=3):
        super().__init__()
        self.patch_size = patch_size
        self.temporal_patch_size = temporal_patch_size
        self.in_channels = in_channels
        
        # The magic: Conv3D with kernel = stride = patch_size
        # This means non-overlapping patches!
        self.proj = nn.Conv3d(
            in_channels=in_channels,           # 3 (RGB)
            out_channels=n_embed,              # 1280 (embedding dim)
            kernel_size=(temporal_patch_size, patch_size, patch_size),  # (2, 16, 16)
            stride=(temporal_patch_size, patch_size, patch_size),       # Same as kernel!
            bias=True
        )
    
    def forward(self, x):
        """
        x: (N, 1536) - flattened patches
        returns: (N, n_embed) - embedded patches
        """
        # Reshape flattened patches back to 3D structure
        # (N, 1536) → (N, 3, 2, 16, 16)
        x = x.view(-1, self.in_channels, self.temporal_patch_size, 
                   self.patch_size, self.patch_size)
        
        # Conv3d magic: (N, 3, 2, 16, 16) → (N, n_embed, 1, 1, 1)
        x = self.proj(x)
        
        # Flatten to get embeddings: (N, n_embed, 1, 1, 1) → (N, n_embed)
        x = x.view(-1, x.shape[1])
        return x

# Demo
print("PatchEmbed Demo:")
print("="*50)
patch_embed = PatchEmbed(n_embed=1280)

# Simulate 100 flattened patches (from an image)
fake_patches = torch.randn(100, 3 * 2 * 16 * 16)  # (100, 1536)
embeddings = patch_embed(fake_patches)

print(f"Input patches shape:  {fake_patches.shape}  (N patches, each 1536 = 3×2×16×16)")
print(f"Output embeds shape:  {embeddings.shape}  (N patches, each 1280-dim embedding)")
print(f"\nTotal parameters: {sum(p.numel() for p in patch_embed.parameters()):,}")

## Part 4: Rotary Position Embeddings (RoPE)

This is one of the most elegant ideas in modern transformers. Instead of *adding* position information, we *rotate* query and key vectors.

**Intuition**: Imagine each position as a rotation angle. Nearby positions have similar angles, so their dot products are naturally higher. The rotation is applied in 2D subspaces of the embedding.

In [ ]:
# ============================================================================
# ROTARY POSITION EMBEDDINGS (RoPE)
# ============================================================================

class RotaryEmbedding(nn.Module):
    """
    RoPE: Rotary Position Embedding
    
    Key insight: We encode position by ROTATING vectors, not adding to them.
    
    For each pair of dimensions, we rotate by an angle proportional to:
    - position (further = more rotation)
    - dimension index (higher dims rotate slower)
    """
    def __init__(self, dim, theta=10000.0):
        super().__init__()
        # Compute inverse frequencies for each dimension pair
        # Lower dims: high frequency (fast rotation)
        # Higher dims: low frequency (slow rotation)
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
    
    def forward(self, seq_len):
        """
        Generate rotation angles for each position.
        Returns: (seq_len, dim/2) - angles for each position and dimension pair
        """
        positions = torch.arange(seq_len, device=self.inv_freq.device)
        # Outer product: position × frequency
        freqs = torch.outer(positions, self.inv_freq)
        return freqs

def rotate_half(x):
    """Split and rotate: [x1, x2] → [-x2, x1]"""
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, freqs):
    """
    Apply rotation to query and key vectors.
    
    The rotation formula:
    x_rotated = x * cos(θ) + rotate_half(x) * sin(θ)
    """
    cos = freqs.cos().unsqueeze(0).unsqueeze(0)  # (1, 1, seq, dim/2)
    sin = freqs.sin().unsqueeze(0).unsqueeze(0)
    
    # Double up cos/sin to match embedding dim
    cos = torch.cat([cos, cos], dim=-1)
    sin = torch.cat([sin, sin], dim=-1)
    
    q_rotated = (q * cos) + (rotate_half(q) * sin)
    k_rotated = (k * cos) + (rotate_half(k) * sin)
    return q_rotated, k_rotated

# Visualize RoPE frequencies
print("RoPE Visualization:")
print("="*50)

rope = RotaryEmbedding(dim=64)
freqs = rope(seq_len=100)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Show how different dimensions rotate at different speeds
for i, dim_idx in enumerate([0, 10, 20, 31]):
    axes[0].plot(freqs[:, dim_idx].numpy(), label=f'dim {dim_idx*2}')
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Rotation Angle')
axes[0].set_title('Rotation angles by position (different dims)')
axes[0].legend()

# Show the full frequency map
im = axes[1].imshow(freqs.numpy().T, aspect='auto', cmap='viridis')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Dimension pair')
axes[1].set_title('Full RoPE frequency map')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

## Part 5: Vision Attention with 2D RoPE

Here's where vision and language models differ. In language, positions are 1D (token 0, 1, 2...). 

In vision, we have **2D positions** (row, column). We apply RoPE separately to height and width positions!

In [ ]:
# ============================================================================
# VISION ATTENTION - 2D Positional Encoding
# ============================================================================

class VisionAttention(nn.Module):
    """
    Multi-head attention for vision, with 2D RoPE.
    
    Key difference from language attention:
    - We encode (height, width) positions separately
    - No causal mask (images aren't autoregressive)
    - But we DO mask between different images in a batch
    """
    def __init__(self, n_embed=1280, n_heads=16):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = n_embed // n_heads
        
        # Single projection for Q, K, V
        self.qkv = nn.Linear(n_embed, n_embed * 3)
        self.proj = nn.Linear(n_embed, n_embed)
    
    def forward(self, x, rotary_pos_emb):
        """
        x: (seq_len, n_embed)  - note: no batch dim, patches are concatenated
        rotary_pos_emb: (seq_len, head_dim) - 2D position encodings
        """
        seq_len = x.shape[0]
        
        # Project to Q, K, V
        qkv = self.qkv(x)  # (seq_len, 3 * n_embed)
        qkv = qkv.reshape(seq_len, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(1, 0, 2, 3).unbind(0)  # each: (seq_len, n_heads, head_dim)
        
        # Apply 2D RoPE
        q = self._apply_rope(q, rotary_pos_emb)
        k = self._apply_rope(k, rotary_pos_emb)
        
        # Standard attention
        q = q.transpose(0, 1)  # (n_heads, seq_len, head_dim)
        k = k.transpose(0, 1)
        v = v.transpose(0, 1)
        
        attn = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)
        
        out = out.transpose(0, 1).reshape(seq_len, -1)
        return self.proj(out)
    
    def _apply_rope(self, x, freqs):
        """Apply rotary embeddings to query or key."""
        cos = freqs.cos().unsqueeze(1)  # (seq_len, 1, head_dim/2)
        sin = freqs.sin().unsqueeze(1)
        cos = torch.cat([cos, cos], dim=-1)  # (seq_len, 1, head_dim)
        sin = torch.cat([sin, sin], dim=-1)
        return (x * cos) + (rotate_half(x) * sin)

print("Vision Attention Demo:")
print("="*50)

attn = VisionAttention(n_embed=256, n_heads=8)
x = torch.randn(100, 256)  # 100 patches, 256-dim embeddings
pos_emb = torch.randn(100, 16)  # 2D position encodings (simplified)

out = attn(x, pos_emb)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")

## Part 6: The MLP (Feed-Forward Network)

Every transformer block has an MLP after attention. Qwen3-VL uses **SwiGLU** (like LLaMA):

```
MLP(x) = down_proj(SiLU(gate_proj(x)) * up_proj(x))
```

The key insight: we have TWO up-projections (gate and up), and we multiply their outputs together. This is the "gating" mechanism.

In [ ]:
# ============================================================================
# MLP with SwiGLU activation
# ============================================================================

class DenseMLP(nn.Module):
    """
    SwiGLU MLP used in both vision encoder and language model.
    
    Pattern:
    x → [gate_proj + SiLU] * [up_proj] → down_proj → output
    
    The gating (multiplication) is what makes this "GLU" - Gated Linear Unit.
    SiLU (swish) activation: x * sigmoid(x)
    """
    def __init__(self, n_embed, n_mlp):
        super().__init__()
        self.gate_proj = nn.Linear(n_embed, n_mlp, bias=False)
        self.up_proj = nn.Linear(n_embed, n_mlp, bias=False)
        self.down_proj = nn.Linear(n_mlp, n_embed, bias=False)
    
    def forward(self, x):
        # The SwiGLU formula:
        # 1. gate_proj(x) → apply SiLU → this is the "gate"
        # 2. up_proj(x) → this is the "value"
        # 3. Multiply gate * value → this is the gating!
        # 4. down_proj to get back to n_embed dims
        gate = F.silu(self.gate_proj(x))  # (B, T, n_mlp)
        up = self.up_proj(x)               # (B, T, n_mlp)
        return self.down_proj(gate * up)   # (B, T, n_embed)

# Demo
print("SwiGLU MLP Demo:")
print("="*50)

mlp = DenseMLP(n_embed=256, n_mlp=1024)  # 4× expansion ratio
x = torch.randn(2, 10, 256)  # (batch=2, seq=10, embed=256)
out = mlp(x)

print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"\nExpansion: {256} → {1024} → {256}")
print(f"Parameters: {sum(p.numel() for p in mlp.parameters()):,}")

## Part 7: The Patch Merger

After the vision encoder processes all patches, we need to merge them and project to match the language model's embedding dimension.

The patch merger does 2×2 spatial merging (4 patches → 1 token) to reduce sequence length.

In [ ]:
# ============================================================================
# PATCH MERGER - Compressing vision representations
# ============================================================================

class PatchMerger(nn.Module):
    """
    Merges 2×2 spatial patches into single tokens.
    
    Why? To reduce sequence length by 4×! Otherwise vision tokens would
    dominate the context window.
    
    For a 672×448 image:
    - Before merge: 42×28 = 1176 patches
    - After merge:  21×14 = 294 tokens
    """
    def __init__(self, n_embed=1280, n_output=1536, spatial_merge_size=2):
        super().__init__()
        # After merging 2×2 patches, we have 4× the embedding dim
        hidden_size = n_embed * (spatial_merge_size ** 2)  # 1280 × 4 = 5120
        
        self.norm = nn.LayerNorm(n_embed)
        self.fc1 = nn.Linear(hidden_size, hidden_size)  # 5120 → 5120
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_size, n_output)     # 5120 → 1536
    
    def forward(self, x):
        """
        x: (N, n_embed) where N = grid_h × grid_w × grid_t
        
        We reshape so that each output token sees 4 input patches.
        """
        # Normalize
        x = self.norm(x)
        
        # Reshape: group 4 patches into 1
        # This flattening happens in the vision encoder's forward
        # Here we just project
        x = self.fc2(self.act(self.fc1(x)))
        return x

print("Patch Merger Demo:")
print("="*50)
print("After 2×2 merge, 4 patches (each 1280-dim) become 1 token (1536-dim)")
print("Input:  4 × 1280 = 5120 dims")
print("Output: 1536 dims (matches language model embedding)")

## Part 8: M-RoPE (Multimodal Rotary Position Embedding)

This is Qwen3-VL's secret sauce for handling both text and images. Instead of 1D positions, we use **3D positions**: (temporal, height, width).

For text tokens: all three dimensions have the same sequential position.
For vision tokens: each dimension encodes spatial/temporal location within the image.

In [ ]:
# ============================================================================
# M-RoPE: Multimodal Rotary Position Embedding
# ============================================================================

def visualize_mrope():
    """
    Visualize how M-RoPE encodes positions for text vs vision tokens.
    
    The key insight: we have 3 position dimensions!
    
    Text token at position 5:
        temporal=5, height=5, width=5  (all same)
    
    Vision token at (row=2, col=3) in image after position 5:
        temporal=5, height=5+2=7, width=5+3=8
    """
    
    # Simulate a sequence: [text, text, IMAGE(3x3), text, text]
    # Positions:           [0,    1,    (image),    2,    3]
    
    print("M-RoPE Position Assignment:")
    print("="*60)
    print("\nSequence: [Hello] [World] [IMAGE 3×3] [What's] [this?]")
    print("\n" + "-"*60)
    print(f"{'Token':<15} {'T (temporal)':<15} {'H (height)':<15} {'W (width)':<15}")
    print("-"*60)
    
    # Text tokens before image
    print(f"{'Hello':<15} {'0':<15} {'0':<15} {'0':<15}")
    print(f"{'World':<15} {'1':<15} {'1':<15} {'1':<15}")
    
    # Vision tokens (3×3 grid at position 2)
    image_start = 2
    for row in range(3):
        for col in range(3):
            token_name = f"img[{row},{col}]"
            t = image_start
            h = image_start + row
            w = image_start + col
            print(f"{token_name:<15} {t:<15} {h:<15} {w:<15}")
    
    # Text tokens after image
    print(f"{'Whats':<15} {'3':<15} {'3':<15} {'3':<15}")
    print(f"{'this?':<15} {'4':<15} {'4':<15} {'4':<15}")
    
    print("\n" + "="*60)
    print("Notice: Vision tokens share temporal position but differ in H/W!")
    print("This lets the model understand spatial relationships in images.")

visualize_mrope()

## Part 9: Mixture of Experts (MoE)

Qwen3-VL can use Mixture of Experts in the language model. Instead of one big MLP, we have many smaller "expert" MLPs, and a router decides which experts to use for each token.

This gives us **more parameters with less compute** - we only use a subset of experts per token.

In [ ]:
# ============================================================================
# MIXTURE OF EXPERTS (MoE)
# ============================================================================

class MoEMLP(nn.Module):
    """
    Mixture of Experts MLP.
    
    Instead of one MLP, we have N experts. For each token:
    1. Router decides top-k experts to use
    2. Run token through selected experts
    3. Weighted sum of expert outputs
    
    Benefits:
    - More total parameters (= more capacity)
    - Same compute per token (only k experts run)
    """
    def __init__(self, n_embed, n_mlp, n_experts=8, top_k=2):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        
        # Router: decides which experts to use
        self.gate = nn.Linear(n_embed, n_experts, bias=False)
        
        # Individual experts (simplified - real impl shares weights)
        self.experts = nn.ModuleList([
            DenseMLP(n_embed, n_mlp) for _ in range(n_experts)
        ])
    
    def forward(self, x):
        B, T, _ = x.shape
        x_flat = x.reshape(-1, x.shape[-1])  # (B*T, n_embed)
        
        # Route: which experts for each token?
        router_logits = self.gate(x_flat)  # (B*T, n_experts)
        routing_weights = F.softmax(router_logits, dim=-1)
        
        # Select top-k experts
        topk_weights, topk_indices = torch.topk(routing_weights, self.top_k, dim=-1)
        topk_weights = topk_weights / topk_weights.sum(dim=-1, keepdim=True)  # normalize
        
        # Run through selected experts and combine (simplified)
        output = torch.zeros_like(x_flat)
        for i in range(self.top_k):
            expert_idx = topk_indices[:, i]
            for e in range(self.n_experts):
                mask = expert_idx == e
                if mask.any():
                    output[mask] += topk_weights[mask, i:i+1] * self.experts[e](x_flat[mask])
        
        return output.view(B, T, -1)

# Demo
print("MoE Demo:")
print("="*50)
moe = MoEMLP(n_embed=256, n_mlp=512, n_experts=8, top_k=2)
x = torch.randn(2, 10, 256)
out = moe(x)

print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"\n8 experts, but only 2 active per token!")
print(f"Total params: {sum(p.numel() for p in moe.parameters()):,}")
print(f"Active params per token: ~{sum(p.numel() for p in moe.experts[0].parameters()) * 2:,}")

## Part 10: Complete Forward Pass Walkthrough

Let's trace a complete forward pass through Qwen3-VL. We'll track shapes at every step.

**Input**: "What's in this image?" + a 672×448 image

In [ ]:
# ============================================================================
# COMPLETE FORWARD PASS WALKTHROUGH
# ============================================================================

def trace_forward_pass():
    """
    Trace the complete forward pass with example shapes.
    
    Example: Image (672×448) + "What's in this image?"
    """
    print("\n" + "="*70)
    print(" QWEN3-VL FORWARD PASS")
    print("="*70)
    
    # -------------------- INPUT PROCESSING --------------------
    print("\n📥 INPUT PROCESSING")
    print("-"*50)
    
    # Text tokenization
    print("\nText: 'What's in this image?'")
    print("  → Tokenize → [token_ids]")
    print("  → Add image placeholders")
    print("  → input_ids shape: (1, ~344)")  # ~50 text + 294 vision
    
    # Image processing
    print("\nImage: (672, 448, 3)")
    print("  → Resize to grid-aligned: (672, 448)")
    print("  → Normalize: (pixel - 0.5) / 0.5")
    print("  → Transpose: (3, 672, 448)")
    print("  → Temporal tile: (2, 3, 672, 448)")  # Duplicate for temporal
    print("  → Extract patches: (1176, 1536)")  # 42×28 patches, each 3×2×16×16
    print("  → d_image: [1, 42, 28]  (grid dimensions)")
    
    # -------------------- VISION ENCODER --------------------
    print("\n\n🔮 VISION ENCODER")
    print("-"*50)
    
    print("\nPatchEmbed (Conv3d):")
    print("  (1176, 1536) → reshape → (1176, 3, 2, 16, 16)")
    print("  → Conv3d → (1176, 1280)  [hidden_states]")
    
    print("\nPosition Embeddings:")
    print("  Learnable pos_embed: (1176, 1280)")
    print("  hidden_states = hidden_states + pos_embed")
    
    print("\nVision Blocks (×32 layers):")
    print("  Each block: LayerNorm → Attention → Add → LayerNorm → MLP → Add")
    print("  Shape preserved: (1176, 1280)")
    
    print("\nPatch Merger (2×2 spatial merge):")
    print("  (1176, 1280) → reshape to 2×2 groups")
    print("  → (294, 5120)  [4 patches × 1280 = 5120]")
    print("  → MLP → (294, 1536)  [vision_embed]")
    
    # -------------------- LANGUAGE MODEL --------------------
    print("\n\n🧠 LANGUAGE MODEL")
    print("-"*50)
    
    print("\nToken Embedding:")
    print("  input_ids (1, ~344) → embed_tokens → (1, ~344, 1536)")
    
    print("\nMerge Vision + Text:")
    print("  text_embeds (1, ~344, 1536)")
    print("  vision_embed (294, 1536)")
    print("  → Replace image_pad positions with vision_embed")
    print("  → merged_embeds (1, ~344, 1536)")
    
    print("\nM-RoPE Position IDs:")
    print("  position_ids shape: (3, 1, ~344)  [temporal, height, width]")
    
    print("\nTransformer Blocks (×28 layers):")
    print("  Each block:")
    print("    RMSNorm → Self-Attention (with M-RoPE) → Add")
    print("    RMSNorm → MLP (Dense or MoE) → Add")
    print("  Shape preserved: (1, ~344, 1536)")
    
    print("\nDeepStack Residuals (at specific layers):")
    print("  Add intermediate vision features at vision token positions")
    
    print("\nFinal Norm:")
    print("  RMSNorm → (1, ~344, 1536)")
    
    print("\nLM Head:")
    print("  (1, ~344, 1536) → Linear → (1, ~344, 151936)  [logits]")
    
    # -------------------- GENERATION --------------------
    print("\n\n📤 GENERATION")
    print("-"*50)
    
    print("\nNext Token:")
    print("  logits[:, -1, :] → (1, 151936)")
    print("  → softmax → argmax → token_id")
    print("  → decode → 'The'")
    
    print("\nAutoregressive loop:")
    print("  Append token → run forward → get next token → repeat")
    print("  Stop at <|im_end|> or max_tokens")
    
    print("\n" + "="*70)

trace_forward_pass()

## Part 11: Loading and Running the Real Model

Now let's load the actual Qwen3-VL model and run inference!

In [ ]:
# ============================================================================
# LOAD AND RUN THE REAL MODEL
# ============================================================================

from huggingface_hub import snapshot_download
from model.processor import Processor
from model.model import Qwen3VL

def load_model(model_name="Qwen/Qwen3-VL-2B-Instruct"):
    """Load Qwen3-VL model."""
    print(f"Loading {model_name}...")
    
    # Download model weights
    weights_path = snapshot_download(repo_id=model_name, cache_dir=".cache")
    
    # Load processor (tokenizer + image processor)
    processor = Processor.from_pretrained(model_name)
    
    # Load model
    model = Qwen3VL.from_pretrained(weights_path=weights_path, device_map="auto")
    model.eval()
    
    print(f"Model loaded! Device: {next(model.parameters()).device}")
    return model, processor

# Uncomment to load:
# model, processor = load_model()

In [ ]:
# ============================================================================
# RUN INFERENCE
# ============================================================================

def run_inference(model, processor, image_path, prompt):
    """Run inference on an image with a text prompt."""
    
    # Prepare messages in OpenAI format
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": prompt}
            ]
        }
    ]
    
    # Process inputs
    device = next(model.parameters()).device
    inputs = processor(messages, add_generation_prompt=True, device=device)
    
    print(f"\nInput shapes:")
    print(f"  input_ids: {inputs['input_ids'].shape}")
    print(f"  pixels: {inputs['pixels'].shape if inputs['pixels'] is not None else None}")
    print(f"  d_image: {inputs['d_image'].tolist() if inputs['d_image'] is not None else None}")
    
    # Generate response
    print(f"\nGenerating...")
    output_ids = model.generate(
        input_ids=inputs['input_ids'],
        pixels=inputs['pixels'],
        d_image=inputs['d_image'],
        max_new_tokens=256
    )
    
    # Decode response
    new_tokens = output_ids[0, inputs['input_ids'].shape[1]:]
    response = processor.tokenizer.decode(new_tokens.tolist())
    
    return response

# Example usage (uncomment when model is loaded):
# response = run_inference(model, processor, "path/to/image.jpg", "What's in this image?")
# print(f"\nResponse: {response}")

## Summary: The Key Ideas

1. **Images as Tokens**: Chop image into 16×16 patches, embed them like text tokens

2. **Spatial Merge**: Reduce sequence length by merging 2×2 patches into 1 token

3. **3D Positions (M-RoPE)**: Use (temporal, height, width) positions for multimodal

4. **DeepStack**: Inject intermediate vision features into language model layers

5. **MoE (optional)**: Use multiple expert MLPs for more capacity with same compute

6. **Unified Architecture**: Vision encoder outputs directly replace placeholder tokens - no cross-attention needed!

---

*That's Qwen3-VL! A beautiful, unified architecture for understanding images and text.* 🚀